In [1]:
import sys
sys.path.append('../')
import pickle
import numpy as np
import pandas as pd
import xgboost as xgb
from utils.helper_functions import read_csv_incl_timeindex, smape
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import json

In [ ]:
models = []
periods = [('2018-01-01', '2023-12-31')]

countries = ['FR', 'ES']
for country in countries:
    if country == 'FR':
        targets = ['FR_price', 'FR_export']
    elif country == 'ES':
        targets = ['ES_price']
    for target in targets:
        for start_date, end_date in periods:
            model_name = '/xgb_{}_start_{}_end_{}_best'.format(target, start_date, end_date)
            directory = '../models'
            
            best_params_path= '/xgb_{}_start_{}_end_{}'.format(target, start_date, end_date)
            with open('{}/{}_best_hyperparameters.pkl'.format(directory, best_params_path), 'rb') as f:
                best_parameters = pickle.load(f)
            model = xgb.Booster()

            model.load_model("../models/{}.json".format(model_name))

            params = json.loads(model.save_config())
            print(params)

            models.append(model)

            X_test = read_csv_incl_timeindex('../data/X_test_features_target_xgb_{}_start_{}_end_{}.csv'.format(target, start_date, end_date))
            X_train = read_csv_incl_timeindex('../data/X_train_features_target_xgb_{}_start_{}_end_{}.csv'.format(target, start_date, end_date))


            y_test = read_csv_incl_timeindex('../data/y_test_xgb_{}_start_{}_end_{}.csv'.format(target, start_date, end_date))
            y_train = read_csv_incl_timeindex('../data/y_train_xgb_{}_start_{}_end_{}.csv'.format(target, start_date, end_date))

            y_pred = pd.DataFrame(index=y_test.index)
            y_pred['xgb_pred'] = model.predict(xgb.DMatrix(X_test))
            r2_score_xgb = r2_score(y_test, y_pred['xgb_pred'])

            y_pred_train = pd.DataFrame(index=y_train.index)
            y_pred_train['xgb_pred'] = model.predict(xgb.DMatrix(X_train))
            r2_score_xgb_train = r2_score(y_train, y_pred_train['xgb_pred'])  

            mae_xgb = mean_absolute_error(y_test, y_pred['xgb_pred'])
            mse_xgb = mean_squared_error(y_test, y_pred['xgb_pred'])

            mean_abs_dev = np.mean(np.abs(y_test.iloc[:, 0] - np.mean(y_test.iloc[:, 0])))

            smape_xgb = smape(y_test.values.reshape(-1), y_pred['xgb_pred'].values.reshape(-1))
            print(y_pred['xgb_pred'].values.shape, y_test.values.shape)
            
            print(start_date, end_date)
            print("XGB results for target: {}".format(target))
            print("R2: {:0.10f}".format(r2_score_xgb))
            print("R2 train: {:0.2f}".format(r2_score_xgb_train))
            print("MAE: {:0.2f}".format(mae_xgb))
            print("rmse: {:0.2f}".format(np.sqrt(mse_xgb)))
            print("SMAPE: {:0.2f} (symmetric mean absolute percentage error)".format(smape_xgb.item()))
            print("(mean label: {:0.2f})".format(y_test.iloc[:, 0].mean()))
            print("------------------------------")

{'learner': {'generic_param': {'device': 'cpu', 'fail_on_invalid_gpu_id': '0', 'n_jobs': '0', 'nthread': '0', 'random_state': '0', 'seed': '0', 'seed_per_iteration': '0', 'validate_parameters': '1'}, 'gradient_booster': {'gbtree_model_param': {'num_parallel_tree': '1', 'num_trees': '1055'}, 'gbtree_train_param': {'process_type': 'default', 'tree_method': 'auto', 'updater': 'grow_quantile_histmaker', 'updater_seq': 'grow_quantile_histmaker'}, 'name': 'gbtree', 'specified_updater': False, 'tree_train_param': {'alpha': '0', 'cache_opt': '1', 'colsample_bylevel': '1', 'colsample_bynode': '1', 'colsample_bytree': '1', 'eta': '0.300000012', 'gamma': '0', 'grow_policy': 'depthwise', 'interaction_constraints': '', 'lambda': '1', 'learning_rate': '0.300000012', 'max_bin': '256', 'max_cat_threshold': '64', 'max_cat_to_onehot': '4', 'max_delta_step': '0', 'max_depth': '6', 'max_leaves': '0', 'min_child_weight': '1', 'min_split_loss': '0', 'monotone_constraints': '()', 'refresh_leaf': '1', 'reg_al